In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Binary ESI 1 XGBoost Classifier (Combined Raw Vitals + Feature Engineered Flags) (`models/xgboost_raw_esi1_extreme.ipynb`)

This notebook trains a **Binary XGBoost Classifier** for **ESI 1 vs Not ESI 1** combining **Raw Clinical Vitals from `triage_conf.json`** AND **10 Clinical Feature Engineered Predictors** with **Optimal Decision Threshold Tuning**:

### System Architecture & Workflow
1. **Full Feature Set (17 Predictors)**:
   - **7 Raw Predictors (Loaded from `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`.
   - **10 Clinical Feature Engineered Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
2. **Stratified Data Partitioning First**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling to prevent data leakage.
3. **Binary XGBoost Gradient Boosting**: Fits binary decision trees (`objective = "binary:logistic"`, `eval_metric = "logloss"`) via `xgb.DMatrix` and `xgb.train()`.
4. **Validation-Based Optimal Threshold Tuning**: Scans decision thresholds $\tau \in [0.01, 0.95]$ on the **Validation Set** to find the optimal cutoff $\tau^*$ maximizing F1 Score / Youden's J Statistic, overcoming default 0.50 cutoff penalty on rare ESI 1 class.
5. **Comprehensive Benchmarking Across Splits**: Benchmarks performance at both **Default Cutoff (0.50)** and **Optimal Tuned Cutoff ($\tau^*$)** across Train, Validation, and Test sets.
6. **Reports & Artifacts**:
   - **Threshold Tuning Curve Plot**: `plots/xgboost_raw_esi1_threshold_curve.png`.
   - **Metrics Comparison Bar Chart**: `plots/xgboost_raw_esi1_metrics_barchart.png`.
   - **CSV Reports**: `reports/xgboost_raw_esi1_val_report.csv`, `reports/xgboost_raw_esi1_test_report.csv`.
   - **Model Export**: Saved to `deploy/xgboost_raw_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(tidyr)
library(pROC)
library(xgboost)
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Raw Features:    ", paste(config$features$data_name, collapse = ", "), "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Reinsert 10 FE Flags Alongside 7 Raw Features (17 Predictors Total)
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 17 Predictors: 7 Raw Vitals/Demographics + 10 Clinical FE Vital Anomaly Flags
df_full <- data.frame(
  # 7 Raw Features from triage_conf.json
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = raw_df$triage_vital_hr,
  triage_vital_sbp        = raw_df$triage_vital_sbp,
  triage_vital_rr         = raw_df$triage_vital_rr,
  triage_vital_o2         = raw_df$triage_vital_o2,
  # 10 Reinserted Feature Engineered Flags
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (17 Total):\n")
print(setdiff(names(df_full), "target_layer1"))
cat("\nNatural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Sample Count Regulation
# ---------------------------------------------------------
set.seed(config$training$random_state)
target_not_1_count <- NULL
sample_ratio_not_1 <- 3.2   # Ratio of 'not_1' samples relative to ESI 1 count
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Standardize continuous vital signs and age
cont_cols <- intersect(c("age", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2"), names(train_df))
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
idx_1_tr     <- which(train_df$target_layer1 == "1")
idx_not_1_tr <- which(train_df$target_layer1 == "not_1")
n_esi1_tr    <- length(idx_1_tr)
n_not_1_keep  <- if (!is.null(target_not_1_count)) min(target_not_1_count, length(idx_not_1_tr)) else min(as.integer(n_esi1_tr * sample_ratio_not_1), length(idx_not_1_tr))
kept_not_1_tr <- sample(idx_not_1_tr, size = n_not_1_keep)
kept_1_tr     <- idx_1_tr
train_df <- train_df[sort(c(kept_1_tr, kept_not_1_tr)), ]
cat(sprintf("Training Sample Count Regulation Applied (ESI 1: %d | 'not_1': %d | Total: %d)\n\n",
            length(kept_1_tr), length(kept_not_1_tr), nrow(train_df)))
feat_names <- setdiff(names(train_df), "target_layer1")
X_train <- as.matrix(train_df[, feat_names])
y_train <- ifelse(train_df$target_layer1 == "1", 1, 0)
X_val   <- as.matrix(val_df[, feat_names])
y_val   <- ifelse(val_df$target_layer1 == "1", 1, 0)
X_test  <- as.matrix(test_df[, feat_names])
y_test  <- ifelse(test_df$target_layer1 == "1", 1, 0)
dtrain_xgb <- xgb.DMatrix(data = X_train, label = y_train)
dval_xgb   <- xgb.DMatrix(data = X_val, label = y_val)
dtest_xgb  <- xgb.DMatrix(data = X_test, label = y_test)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary XGBoost Model on Combined Feature Set (17 Predictors)
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training Binary XGBoost Model on 17 Features (ESI 1 vs Not ESI 1)...\n")
xgb_params <- list(
  objective        = "binary:logistic",
  eval_metric      = "logloss",
  eta              = 0.05,
  max_depth        = 6,
  subsample        = 0.8,
  colsample_bytree = 0.8
)
xgb_model <- xgb.train(
  params    = xgb_params,
  data      = dtrain_xgb,
  nrounds   = 150,
  evals     = list(train = dtrain_xgb, val = dval_xgb),
  early_stopping_rounds = 20,
  verbose   = 0
)
cat("Combined Feature Binary XGBoost ESI 1 Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Validation-Based Optimal Decision Threshold Tuning
# ---------------------------------------------------------
prob_tr_1   <- predict(xgb_model, newdata = dtrain_xgb)
prob_val_1  <- predict(xgb_model, newdata = dval_xgb)
prob_test_1 <- predict(xgb_model, newdata = dtest_xgb)
find_optimal_threshold <- function(actual_factor, prob_positive, method = "f1") {
  thresholds <- seq(0.01, 0.90, by = 0.005)
  act_bin <- ifelse(actual_factor == "1", 1, 0)
  
  best_tau <- 0.50
  best_score <- -1
  grid_df <- data.frame()
  
  for (tau in thresholds) {
    pred_bin <- ifelse(prob_positive >= tau, 1, 0)
    
    tp <- sum(pred_bin == 1 & act_bin == 1)
    fp <- sum(pred_bin == 1 & act_bin == 0)
    fn <- sum(pred_bin == 0 & act_bin == 1)
    tn <- sum(pred_bin == 0 & act_bin == 0)
    
    prec <- ifelse((tp + fp) > 0, tp / (tp + fp), 0)
    rec  <- ifelse((tp + fn) > 0, tp / (tp + fn), 0)
    spec <- ifelse((tn + fp) > 0, tn / (tn + fp), 0)
    f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
    youden <- rec + spec - 1
    
    grid_df <- rbind(grid_df, data.frame(Threshold = tau, Precision = prec, Recall = rec, Specificity = spec, F1 = f1, Youden = youden))
    
    score <- if (method == "f1") f1 else youden
    if (score > best_score) {
      best_score <- score
      best_tau   <- tau
    }
  }
  return(list(best_tau = best_tau, best_score = best_score, grid_df = grid_df))
}
tune_res <- find_optimal_threshold(val_df$target_layer1, prob_val_1, method = "f1")
opt_tau  <- tune_res$best_tau
cat(sprintf("============================================================\n"))
cat(sprintf("   OPTIMAL THRESHOLD TUNING COMPLETED (VALIDATION SET)\n"))
cat(sprintf("============================================================\n"))
cat(sprintf("  Default Cutoff Threshold : 0.5000\n"))
cat(sprintf("  Optimal Cutoff Threshold : %.4f (Maximizing Validation F1 Score)\n", opt_tau))
cat(sprintf("  Max Validation F1 Score  : %.4f\n", tune_res$best_score))
cat(sprintf("============================================================\n\n"))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Comprehensive Benchmark (Default Cutoff 0.50 vs Optimal Cutoff tau*)
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_with_cutoff <- function(prob_1, actual_factor, cutoff, set_name) {
  pred_val <- ifelse(prob_1 >= cutoff, "1", "not_1")
  pred_fac <- factor(pred_val, levels = c("1", "not_1"))
  act_fac  <- factor(actual_factor, levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  if (is.na(prec)) prec <- 0
  if (is.na(rec))  rec  <- 0
  f1   <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  pr_auc  <- calc_pr_auc(ifelse(act_fac == "1", 1, 0), prob_1)
  roc_obj <- tryCatch(pROC::roc(act_fac, prob_1, levels = c("not_1", "1")), error = function(e) NULL)
  roc_auc <- if (!is.null(roc_obj)) as.numeric(roc_obj$auc) else NA
  
  actual_table <- table(act_fac)
  pred_table   <- table(pred_fac)
  diff_vec     <- as.numeric(pred_table) - as.numeric(actual_table)
  diff_str     <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = c("1", "not_1"),
    Actual_Count = as.numeric(actual_table),
    Pred_Count   = as.numeric(pred_table),
    Diff         = diff_str,
    Precision    = round(c(prec, ifelse(is.na(cm$byClass["Neg Pred Value"]), 0, cm$byClass["Neg Pred Value"])), 4),
    Recall       = round(c(rec, ifelse(is.na(cm$byClass["Specificity"]), 0, cm$byClass["Specificity"])), 4),
    PR_AUC       = round(c(pr_auc, NA), 4)
  )
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   COMBINED FEATURE ESI 1 XGBOOST - %s SET (Cutoff = %.4f)\n", toupper(set_name), cutoff))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy             : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ESI 1 Precision      : %.4f (%.2f%%)\n", prec, prec * 100))
  cat(sprintf("  ESI 1 Recall (Sens)  : %.4f (%.2f%%)\n", rec, rec * 100))
  cat(sprintf("  ESI 1 F1 Score       : %.4f\n", f1))
  cat(sprintf("  ESI 1 PR-AUC         : %.4f\n", pr_auc))
  cat(sprintf("  ROC-AUC              : %.4f\n", roc_auc))
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, prec = prec, rec = rec, f1 = f1, pr_auc = pr_auc, roc_auc = roc_auc, report_df = report_df))
}
cat("--- BENCHMARK AT DEFAULT CUTOFF (0.50) ---\n")
def_val  <- evaluate_with_cutoff(prob_val_1,  val_df$target_layer1,  0.50, "Validation (Default 0.50)")
def_test <- evaluate_with_cutoff(prob_test_1, test_df$target_layer1, 0.50, "Test (Default 0.50)")
cat("--- BENCHMARK AT OPTIMAL TUNED CUTOFF (tau*) ---\n")
opt_val  <- evaluate_with_cutoff(prob_val_1,  val_df$target_layer1,  opt_tau, sprintf("Validation (Optimal %.4f)", opt_tau))
opt_test <- evaluate_with_cutoff(prob_test_1, test_df$target_layer1, opt_tau, sprintf("Test (Optimal %.4f)", opt_tau))
# Write CSV Reports for Optimal Cutoff
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(opt_val$report_df,  file = file.path(reports_dir, "xgboost_raw_esi1_val_report.csv"),  row.names = FALSE)
write.csv(opt_test$report_df, file = file.path(reports_dir, "xgboost_raw_esi1_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/xgboost_raw_esi1_val_report.csv\n")
cat("Test CSV Report written to:       reports/xgboost_raw_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Diagnostic Plots (Threshold Tuning Curve & Metrics Comparison)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
# 1. THRESHOLD TUNING CURVE PLOT
curve_df <- tune_res$grid_df %>%
  pivot_longer(cols = c("Precision", "Recall", "F1", "Youden"), names_to = "Metric", values_to = "Score")
p_curve <- ggplot(curve_df, aes(x = Threshold, y = Score, color = Metric)) +
  geom_line(size = 1) +
  geom_vline(xintercept = opt_tau, linetype = "dashed", color = "red", size = 0.8) +
  annotate("text", x = opt_tau + 0.05, y = 0.5, label = sprintf("Optimal Cutoff \ntau* = %.4f", opt_tau), color = "red", fontface = "bold") +
  theme_minimal() +
  labs(title = "Optimal Decision Threshold Tuning Curve (Validation Set)",
       subtitle = "Evaluating Precision, Recall, F1 Score, and Youden's J Statistic across Cutoff Thresholds",
       x = "Probability Cutoff Threshold (tau)", y = "Metric Score") +
  theme(plot.title = element_text(face = "bold", size = 12), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_raw_esi1_threshold_curve.png"), plot = p_curve, width = 9, height = 5, dpi = 300)
cat("Threshold Tuning Curve Plot saved to: plots/xgboost_raw_esi1_threshold_curve.png\n")
# 2. DEFAULT VS OPTIMAL CUTOFF METRICS COMPARISON BAR CHART
metrics_summary <- data.frame(
  Evaluation  = factor(c("Default (0.50)", "Optimal Tuned (tau*)"), levels = c("Default (0.50)", "Optimal Tuned (tau*)")),
  Accuracy    = c(def_test$acc,  opt_test$acc),
  Precision   = c(def_test$prec, opt_test$prec),
  Recall      = c(def_test$rec,  opt_test$rec),
  F1_Score    = c(def_test$f1,   opt_test$f1),
  PR_AUC      = c(def_test$pr_auc, opt_test$pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Precision", "Recall", "F1_Score", "PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Evaluation)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Default (0.50)" = "#e07a5f", "Optimal Tuned (tau*)" = "#2b5c8f")) +
  labs(title = "Default Cutoff (0.50) vs. Optimal Tuned Cutoff Metrics Comparison (Test Set)",
       subtitle = "Demonstrating performance gains from Optimal Threshold Calibration",
       y = "Metric Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "xgboost_raw_esi1_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/xgboost_raw_esi1_metrics_barchart.png\n")
print(p_curve)
print(p_bar)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 8: Save Combined Feature XGBoost ESI 1 Model & Optimal Cutoff Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "xgboost_raw_esi1_extreme_model.rds")
saveRDS(list(model = xgb_model, preproc = preproc, opt_threshold = opt_tau), file = model_path)
cat(sprintf("Combined Feature Binary ESI 1 XGBoost model & Optimal Cutoff (tau* = %.4f) saved to: %s\n", opt_tau, model_path))